# INDEX
### **1. Data loading**
### **2. xresnet1d101 for ECG signal only on PTB-XL**
### **3. Fully connected layers for metadata only on PTB-XL**
### **4. Fusion model**


# 1. Data loading

In [13]:
# Install dependencies
!pip install wfdb

In [14]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/ptbxl.zip"
extract_path = "/content/ptbxl"

# Only unzip if not already extracted in this session
if not os.path.exists(extract_path):
    print("Copying zip to local storage...")
    !cp "{zip_path}" /content/

    print("Unzipping...")
    with zipfile.ZipFile("/content/ptbxl.zip", 'r') as zip_ref:
        zip_ref.extractall("/content/")

    print("Done.")
else:
    print("Dataset already available in local session.")

Copying zip to local storage...
Unzipping...
Done.


In [16]:
path = "/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3/"

In [17]:
import pandas as pd
import numpy as np
import wfdb
import ast
import os
import matplotlib.pyplot as plt

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path + f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path + f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

sampling_rate = 100

# Load and convert annotation data
Y = pd.read_csv(path + 'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path + 'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

# Split data into train, validation, and test according to PTB-XL folds
train_folds = list(range(1, 9))   # 1–8
val_fold = 9                      # 9
test_fold = 10                    # 10

# Train
X_train = X[np.where(Y.strat_fold.isin(train_folds))]
y_train = Y[Y.strat_fold.isin(train_folds)].diagnostic_superclass

# Validation
X_val = X[np.where(Y.strat_fold == val_fold)]
y_val = Y[Y.strat_fold == val_fold].diagnostic_superclass

# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

print("X_train shape:", X_train.shape)
print("y_train sample:", y_train.head())

print("X_val shape:", X_val.shape)
print("y_val sample:", y_val.head())

print("X_test shape:", X_test.shape)
print("y_test sample:", y_test.head())


X_train shape: (17418, 1000, 12)
y_train sample: ecg_id
1    [NORM]
2    [NORM]
3    [NORM]
4    [NORM]
5    [NORM]
Name: diagnostic_superclass, dtype: object
X_val shape: (2183, 1000, 12)
y_val sample: ecg_id
8       [MI]
10    [NORM]
17        []
18        []
20        []
Name: diagnostic_superclass, dtype: object
X_test shape: (2198, 1000, 12)
y_test sample: ecg_id
9     [NORM]
38    [NORM]
40    [NORM]
57    [NORM]
59    [NORM]
Name: diagnostic_superclass, dtype: object


In [18]:
folder_path = path + "records100/00000"

# List all .hea files (each one corresponds to one ECG)
records = [f.replace(".hea", "") for f in os.listdir(folder_path) if f.endswith(".hea")]

print("Found records:", records[:10])

Found records: ['00583_lr', '00843_lr', '00623_lr', '00918_lr', '00571_lr', '00074_lr', '00161_lr', '00501_lr', '00653_lr', '00793_lr']


# 2. xresnet1d101 for ECG signal only on PTB-XL

In [19]:
# Step 1: Convert diagnostic superclasses to multi‑hot vectors

In [20]:
from sklearn.preprocessing import MultiLabelBinarizer

classes = sorted(list(set([c for sub in y_train for c in sub])))
print("Diagnostic superclasses:", classes)

mlb = MultiLabelBinarizer(classes=classes)

y_train_bin = mlb.fit_transform(y_train)
y_val_bin   = mlb.transform(y_val)
y_test_bin  = mlb.transform(y_test)

print("y_train_bin shape:", y_train_bin.shape)

Diagnostic superclasses: ['CD', 'HYP', 'MI', 'NORM', 'STTC']
y_train_bin shape: (17418, 5)


In [21]:
# Step 2: Build a PyTorch Dataset

#  ECGs are shaped (N, 1000, 12) but PyTorch expects (N, 12, 1000)

import torch
from torch.utils.data import Dataset, DataLoader

class PTBXL_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)  # (N, 12, 1000)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = PTBXL_Dataset(X_train, y_train_bin)
val_ds   = PTBXL_Dataset(X_val, y_val_bin)
test_ds  = PTBXL_Dataset(X_test, y_test_bin)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl   = DataLoader(val_ds, batch_size=32)
test_dl  = DataLoader(test_ds, batch_size=32)

In [22]:
import torch
torch.cuda.is_available()

False

In [23]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [24]:
# Step 3: Import xresnet1d101 from the repo

# Copying the repo folder into the Colab environment:
#!git clone https://github.com/helme/ecg_ptbxl_benchmarking.git
# Import xresnet1d101
# Add the parent directory to Python path
#import sys
#sys.path.append("/content/drive/MyDrive/TFG/Code")
# Import the model correctly
#from xresnet1d_clean import xresnet1d101
#model = xresnet1d101(
    #input_channels=12,          # PTB‑XL has 12 leads
   # num_classes=len(classes),   # 5 diagnostic superclasses
   # kernel_size=5,
   # kernel_size_stem=7,
#)
#model = model.cuda()


# -----------------------------
# Similar to the repository XResNet1d101
# -----------------------------

import torch
import torch.nn as nn

# -----------------------------
# Basic building blocks
# -----------------------------

class ConvBNAct(nn.Module):
    """Conv1d + BatchNorm + Activation (ReLU)."""
    def __init__(self, in_ch, out_ch, ks=3, stride=1, act=nn.ReLU):
        super().__init__()
        pad = ks // 2
        self.conv = nn.Conv1d(in_ch, out_ch, ks, stride=stride, padding=pad, bias=False)
        self.bn = nn.BatchNorm1d(out_ch)
        self.act = act(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class ResBlock1d(nn.Module):
    """Official XResNet bottleneck block (expansion=4)."""
    def __init__(self, in_ch, out_ch, stride=1, expansion=4, ks=3):
        super().__init__()

        mid_ch = out_ch // expansion

        # 1x1 → 3x3 → 1x1 bottleneck
        self.conv1 = ConvBNAct(in_ch, mid_ch, ks=1, stride=stride)
        self.conv2 = ConvBNAct(mid_ch, mid_ch, ks=ks, stride=1)
        self.conv3 = nn.Sequential(
            nn.Conv1d(mid_ch, out_ch, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_ch)
        )

        # Shortcut
        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch)
            )

        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out += self.shortcut(x)
        return self.act(out)


# -----------------------------
# XResNet1d backbone
# -----------------------------

class XResNet1d(nn.Module):
    """Faithful to the PTB‑XL repository structure."""
    def __init__(self, block, layers, expansion=4,
                 input_channels=12, num_classes=5,
                 stem_sizes=(32, 32, 64),
                 kernel_size=5, kernel_size_stem=7):
        super().__init__()

        # Stem (3 conv layers + maxpool)
        c1, c2, c3 = stem_sizes
        self.stem = nn.Sequential(
            ConvBNAct(input_channels, c1, ks=kernel_size_stem, stride=2),
            ConvBNAct(c1, c2, ks=kernel_size_stem, stride=1),
            ConvBNAct(c2, c3, ks=kernel_size_stem, stride=1),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )

        # Official block widths
        base = [64, 128, 256, 512]
        block_sizes = [b * expansion for b in base]

        # 4 stages with [3, 4, 23, 3] blocks
        self.layers = nn.Sequential(
            self._make_stage(block, c3, block_sizes[0], layers[0], stride=1, expansion=expansion, ks=kernel_size),
            self._make_stage(block, block_sizes[0], block_sizes[1], layers[1], stride=2, expansion=expansion, ks=kernel_size),
            self._make_stage(block, block_sizes[1], block_sizes[2], layers[2], stride=2, expansion=expansion, ks=kernel_size),
            self._make_stage(block, block_sizes[2], block_sizes[3], layers[3], stride=2, expansion=expansion, ks=kernel_size),
        )

        # Head
        final_ch = block_sizes[3]
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(final_ch, num_classes)
        )

        self._init_weights()

    def _make_stage(self, block, in_ch, out_ch, num_blocks, stride, expansion, ks):
        layers = [block(in_ch, out_ch, stride=stride, expansion=expansion, ks=ks)]
        for _ in range(1, num_blocks):
            layers.append(block(out_ch, out_ch, stride=1, expansion=expansion, ks=ks))
        return nn.Sequential(*layers)

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.stem(x)
        x = self.layers(x)
        x = self.head(x)
        return x


# -----------------------------
# Public constructor
# -----------------------------

def xresnet1d101(input_channels=12, num_classes=5,
                 kernel_size=5, kernel_size_stem=7):
    """XResNet1d‑101: [3, 4, 23, 3] blocks, expansion=4."""
    return XResNet1d(
        block=ResBlock1d,
        layers=[3, 4, 23, 3],
        expansion=4,
        input_channels=input_channels,
        num_classes=num_classes,
        kernel_size=kernel_size,
        kernel_size_stem=kernel_size_stem
    )

# Instantiate the model using the constructor just defined
model = xresnet1d101(
    input_channels=12,
    num_classes=len(classes),
    kernel_size=5,
    kernel_size_stem=7
)

# Move the model to GPU
model = model.cuda()


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
# Step 4: Define loss + optimizer

import torch.nn as nn
import torch.optim as optim

criterion = nn.BCEWithLogitsLoss()    # Because PTB‑XL is multi-label --> BCEWithLogitsLoss
optimizer = optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
# Step 5: Training loop

def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0

    for X, y in loader:
        X, y = X.cuda(), y.cuda()

        optimizer.zero_grad()
        preds = model(X)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for X, y in loader:
            X, y = X.cuda(), y.cuda()
            preds = model(X)
            loss = criterion(preds, y)
            total_loss += loss.item()

    return total_loss / len(loader)

torch.save(model.state_dict(), "ecg_model.pt")


In [ ]:
# Store losses during training
train_losses = []
val_losses = []

# Train for some epochs
for epoch in range(20):
    train_loss = train_epoch(model, train_dl, optimizer, criterion)
    val_loss   = eval_epoch(model, val_dl, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")

# Plot the curves
plt.figure(figsize=(35,10))
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# Compare predictions vs ground truth for one batch

model.eval()
Xb, yb = next(iter(test_dl))
Xb, yb = Xb.cuda(), yb.cuda()

with torch.no_grad():
    preds = torch.sigmoid(model(Xb)).cpu().numpy()

yb = yb.cpu().numpy()

# Show first sample
i = 0
print("Ground truth:", yb[i])
print("Predictions :", preds[i])


plt.figure(figsize=(8,4))
plt.bar(classes, preds[i], alpha=0.6, label="Predicted")
plt.bar(classes, yb[i], alpha=0.6, label="True")
plt.xticks(rotation=45)
plt.ylabel("Probability / Label")
plt.title("Prediction vs Ground Truth")
plt.legend()
plt.show()


In [ ]:
# Compute predictions on the test set

from sklearn.metrics import roc_auc_score, confusion_matrix
import numpy as np

model.eval()

all_preds = []
all_targets = []

with torch.no_grad():
    for X, y in test_dl:
        X = X.cuda()
        preds = torch.sigmoid(model(X)).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(y.numpy())

all_preds = np.vstack(all_preds)
all_targets = np.vstack(all_targets)

print("Predictions shape:", all_preds.shape)
print("Targets shape:", all_targets.shape)


In [ ]:
# AUC per class + macro AUC

auc_scores = {}

for i, cls in enumerate(classes):
    try:
        auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
        auc_scores[cls] = auc
    except:
        auc_scores[cls] = np.nan  # class may be missing in test set

auc_scores

plt.figure(figsize=(8,5))
plt.bar(auc_scores.keys(), auc_scores.values(), color='skyblue')
plt.ylabel("AUC")
plt.title("AUC per Diagnostic Superclass")
plt.ylim(0,1)
plt.grid(axis='y')
plt.show()

macro_auc = np.nanmean(list(auc_scores.values()))
print("Macro AUC:", macro_auc)


In [ ]:
# Confusion matrix

threshold = 0.5
pred_bin = (all_preds >= threshold).astype(int)

import seaborn as sns

for i, cls in enumerate(classes):
    cm = confusion_matrix(all_targets[:, i], pred_bin[:, i])

    plt.figure(figsize=(4,3))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Pred 0", "Pred 1"],
                yticklabels=["True 0", "True 1"])
    plt.title(f"Confusion Matrix – {cls}")
    plt.xlabel("Prediction")
    plt.ylabel("True Label")
    plt.show()


# 3. Fully connected layers for metadata only on PTB-XL